Goal of this notebook: generate measurement cost results for N2. Results will only be obtained for tapered system due to the size of the N2 Hamiltonian.

In [1]:
import numpy as np
import pickle
import scipy.sparse
from utils_ferm import (
    op_action_tz
)
from utils_CSF_and_UCSF import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)
from utils_basic import shift_hamiltonian_qubits
from openfermion import (
    FermionOperator,
    jordan_wigner,
    QubitOperator,
    get_sparse_operator,
    normal_ordered
)

from utils_states import (
    somos_to_seniority_config,
    convert_TZ_format_to_sparse_format,
    variance_of_decomp,
    create_composite_state,
    create_composite_state_prepended,
    expectation,
    matrix_element,
    compress_state,
    convert_dense_format_to_sparse_format,
    variance_of_general_operator
)
from utils_hamiltonian_qubit import (
    process_qubit_hamiltonian_to_remove_irrelevant_terms,
    process_qubit_hamiltonian_to_project_onto_symmetric_subspace
)
from utils_hamiltonian_ferm import (
    process_fermionic_hamiltonian_to_remove_irrelevant_terms
)
from utils_partitioning import (
    sorted_insertion_decomposition,
    augment_decomp_with_pauli_x,
    augment_decomp_with_pauli_x_plus_i_pauli_y
)

filename = f'data/old_data1/n2_data.dump'

with open(filename, 'rb') as f:
    (
    list_CSF,
    list_list_ia_CSF,
    list_list_theta_CSF,
    list_sym_CSF_vec,
    list_UCSF_tz,
    tz_states,
    somos_list,
    psi_GS_UCSF_smik,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

#Rotate orbitals 
if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)


In [2]:
Nqubits       = obt.shape[0]
dim           = 2 ** Nqubits
Norb          = Nqubits // 2
Nstates       = len(tz_states)
fragment_type = 'qwc'

# generate Hamiltonian

Hferm = make_short_H_ferm_op(0, obt, tbt)
Hqub  = jordan_wigner(Hferm)
Hqub -= Hqub.constant
Hqub.compress()

decomp       = sorted_insertion_decomposition(Hqub, fragment_type)
configs      = [somos_to_seniority_config(somos, Norb) for somos in somos_list]
statevectors = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

results = {}

for i in range(Nstates):
    for j in range(Nstates):
        if i >= j:

            # load bra, ket, and swap test state
            
            bra                 = statevectors[i]
            bra_tz              = tz_states[i]
            bra_somos           = somos_list[i]
            bra_config          = configs[i]
            bra_t               = convert_dense_format_to_sparse_format(compress_state(bra.toarray()[0]))

            ket                 = statevectors[j]
            ket_tz              = tz_states[j]
            ket_somos           = somos_list[j]
            ket_config          = configs[j]
            ket_t               = convert_dense_format_to_sparse_format(compress_state(ket.toarray()[0]))

            bra_ket_composite_t = create_composite_state(bra_t, ket_t, Nqubits // 2)

            # taper Hamiltonian

            Hferm_processed = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)
            Hqub_temp       = jordan_wigner(Hferm_processed)
            Hqub_temp      -= Hqub_temp.constant
            Hqub_temp.compress()
            Hqub_tapered    = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(Hqub_temp, Nqubits, bra_config, ket_config)
            Hqub_tapered   -= Hqub_tapered.constant
            Hqub_tapered.compress()
            
            decomp_tapered      = sorted_insertion_decomposition(Hqub_tapered, fragment_type)
            decomp_tapered_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(decomp_tapered, Nqubits // 2)

            if i == j:
                Hqub_tapered_sparse = get_sparse_operator(Hqub_tapered, Nqubits//2)

                var    = variance_of_decomp(decomp_tapered, ket_t, Nqubits // 2, general=True)
                me     = matrix_element(get_sparse_operator(Hqub_tapered, Nqubits // 2), ket_t, ket_t)
                var_lb = variance_of_general_operator(Hqub_tapered_sparse, ket_t).toarray()[0,0]

            else:
                Hqub_tapered_tensor_1q_sparse = get_sparse_operator(
                    Hqub_tapered * (QubitOperator(f'X{Nqubits//2}') + 1j * QubitOperator(f'Y{Nqubits//2}')), Nqubits // 2 + 1
                )

                var    = variance_of_decomp(decomp_tapered_swap, bra_ket_composite_t, Nqubits // 2 + 1, general=True)
                me     = matrix_element(get_sparse_operator(Hqub_tapered, Nqubits // 2), bra_t, ket_t)
                var_lb = variance_of_general_operator(Hqub_tapered_tensor_1q_sparse, bra_ket_composite_t).toarray()[0,0]

            results[(i,j)] = (var, me, var_lb)

            print("{:<2} {:<4} {:<10} {:<15} {:<20}".format(
                i, 
                j, 
                np.round(var, 6), 
                np.round(me, 6), 
                np.round(var_lb, 6)
                ))

0  0    (0.432946+0j) (-43.788244+0j) (0.006769+0j)       
1  0    (0.620623+0j) (0.061907+0j)   (0.015485+0j)       
1  1    (1.864879+0j) (-39.638349+0j) (1.22409+0j)        
2  0    (0.620623+0j) (0.061907+0j)   (0.015485+0j)       
2  1    (154.019973+0j) (-0.775451+0j)  (67.716193+0j)      
2  2    (1.864879+0j) (-39.638349+0j) (1.22409+0j)        
3  0    (0.683238+0j) (0.007746+0j)   (0.001041+0j)       
3  1    (0.361716+0j) (-0.002977+0j)  (0.011391+0j)       
3  2    (0.361716+0j) (-0.002977+0j)  (0.011391+0j)       
3  3    (0.651301+0j) (-41.201957+0j) (0.247805+0j)       
4  0    (0.550216+0j) (-0.001992+0j)  (0.000151+0j)       
4  1    (1.034717+0j) (-0.011198+0j)  (0.010105+0j)       
4  2    (1.075194+0j) (-0.00167+0j)   (0.052273+0j)       
4  3    (0.602803+0j) (0.021849+0j)   (0.022283+0j)       
4  4    (0.697764+0j) (-40.056033+0j) (0.310617+0j)       
5  0    (0.550216+0j) (-0.001992+0j)  (0.000151+0j)       
5  1    (1.075194+0j) (-0.00167+0j)   (0.052273+0j)   

In [ ]:
Nqubits       = obt.shape[0]
dim           = 2 ** Nqubits
Norb          = Nqubits // 2
Nstates       = len(tz_states)
fragment_type = 'qwc'

# generate Hamiltonian

Hferm = make_short_H_ferm_op(0, obt, tbt)
Hqub  = jordan_wigner(Hferm)
Hqub -= Hqub.constant
Hqub.compress()

decomp       = sorted_insertion_decomposition(Hqub, fragment_type)
configs      = [somos_to_seniority_config(somos, Norb) for somos in somos_list]
statevectors = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

results = {}

for i in range(Nstates):
    for j in range(Nstates):
        if i >= j:

            # load bra, ket, and swap test state
            
            bra                 = statevectors[i]
            bra_tz              = tz_states[i]
            bra_somos           = somos_list[i]
            bra_config          = configs[i]
            bra_t               = convert_dense_format_to_sparse_format(compress_state(bra.toarray()[0]))

            ket                 = statevectors[j]
            ket_tz              = tz_states[j]
            ket_somos           = somos_list[j]
            ket_config          = configs[j]
            ket_t               = convert_dense_format_to_sparse_format(compress_state(ket.toarray()[0]))

            bra_ket_composite_t = create_composite_state(bra_t, ket_t, Nqubits // 2)

            # taper Hamiltonian

            Hferm_processed = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)
            Hqub_temp       = jordan_wigner(Hferm_processed)
            Hqub_temp      -= Hqub_temp.constant
            Hqub_temp.compress()
            Hqub_tapered    = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(Hqub_temp, Nqubits, bra_config, ket_config)
            Hqub_tapered   -= Hqub_tapered.constant
            Hqub_tapered.compress()
            
            decomp_tapered      = sorted_insertion_decomposition(Hqub_tapered, fragment_type)
            decomp_tapered_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(decomp_tapered, Nqubits // 2)

            if i == j:
                Hqub_tapered_sparse = get_sparse_operator(Hqub_tapered, Nqubits//2)

                var    = variance_of_decomp(decomp_tapered, ket_t, Nqubits // 2, general=True)
                me     = matrix_element(get_sparse_operator(Hqub_tapered, Nqubits // 2), ket_t, ket_t)
                var_lb = variance_of_general_operator(Hqub_tapered_sparse, ket_t).toarray()[0,0]

            else:
                Hqub_tapered_tensor_1q_sparse = get_sparse_operator(
                    Hqub_tapered * (QubitOperator(f'X{Nqubits//2}') + 1j * QubitOperator(f'Y{Nqubits//2}')), Nqubits // 2 + 1
                )

                var    = variance_of_decomp(decomp_tapered_swap, bra_ket_composite_t, Nqubits // 2 + 1, general=True)
                me     = matrix_element(get_sparse_operator(Hqub_tapered, Nqubits // 2), bra_t, ket_t)
                var_lb = variance_of_general_operator(Hqub_tapered_tensor_1q_sparse, bra_ket_composite_t).toarray()[0,0]

            results[(i,j)] = (var, me, var_lb)

            print("{:<2} {:<4} {:<10} {:<15} {:<20}".format(
                i, 
                j, 
                np.round(var, 6), 
                np.round(me, 6), 
                np.round(var_lb, 6)
                ))

0  0    (0.432946+0j) (-43.788244+0j) (0.006769+0j)       
1  0    (0.620623+0j) (0.061907+0j)   (0.015485+0j)       
1  1    (1.864879+0j) (-39.638349+0j) (1.22409+0j)        
2  0    (0.620623+0j) (0.061907+0j)   (0.015485+0j)       
2  1    (154.019973+0j) (-0.775451+0j)  (67.716193+0j)      
2  2    (1.864879+0j) (-39.638349+0j) (1.22409+0j)        
3  0    (0.683238+0j) (0.007746+0j)   (0.001041+0j)       
3  1    (0.361716+0j) (-0.002977+0j)  (0.011391+0j)       
3  2    (0.361716+0j) (-0.002977+0j)  (0.011391+0j)       
3  3    (0.651301+0j) (-41.201957+0j) (0.247805+0j)       
4  0    (0.550216+0j) (-0.001992+0j)  (0.000151+0j)       
4  1    (1.034717+0j) (-0.011198+0j)  (0.010105+0j)       
4  2    (1.075194+0j) (-0.00167+0j)   (0.052273+0j)       
4  3    (0.602803+0j) (0.021849+0j)   (0.022283+0j)       
4  4    (0.697764+0j) (-40.056033+0j) (0.310617+0j)       
5  0    (0.550216+0j) (-0.001992+0j)  (0.000151+0j)       
5  1    (1.075194+0j) (-0.00167+0j)   (0.052273+0j)   

In [ ]:
sum([(10**6) * np.real(list(results.values())[i][0]) for i in range(len(results.values()))])

In [ ]:
Nqubits       = obt.shape[0]
dim           = 2 ** Nqubits
Norb          = Nqubits // 2
Nstates       = len(tz_states)
fragment_type = 'fc'

# generate Hamiltonian

Hferm = make_short_H_ferm_op(0, obt, tbt)
Hqub  = jordan_wigner(Hferm)
Hqub -= Hqub.constant
Hqub.compress()

decomp       = sorted_insertion_decomposition(Hqub, fragment_type)
configs      = [somos_to_seniority_config(somos, Norb) for somos in somos_list]
statevectors = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

results = {}

for i in range(Nstates):
    for j in range(Nstates):
        if i >= j:

            # load bra, ket, and swap test state
            
            bra                 = statevectors[i]
            bra_tz              = tz_states[i]
            bra_somos           = somos_list[i]
            bra_config          = configs[i]
            bra_t               = convert_dense_format_to_sparse_format(compress_state(bra.toarray()[0]))

            ket                 = statevectors[j]
            ket_tz              = tz_states[j]
            ket_somos           = somos_list[j]
            ket_config          = configs[j]
            ket_t               = convert_dense_format_to_sparse_format(compress_state(ket.toarray()[0]))

            bra_ket_composite_t = create_composite_state(bra_t, ket_t, Nqubits // 2)

            # taper Hamiltonian

            Hferm_processed = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)
            Hqub_temp       = jordan_wigner(Hferm_processed)
            Hqub_temp      -= Hqub_temp.constant
            Hqub_temp.compress()
            Hqub_tapered    = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(Hqub_temp, Nqubits, bra_config, ket_config)
            Hqub_tapered   -= Hqub_tapered.constant
            Hqub_tapered.compress()
            
            decomp_tapered      = sorted_insertion_decomposition(Hqub_tapered, fragment_type)
            decomp_tapered_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(decomp_tapered, Nqubits // 2)

            if i == j:
                var = variance_of_decomp(decomp_tapered, ket_t, Nqubits // 2, general=True)
                me  = matrix_element(get_sparse_operator(Hqub_tapered, Nqubits // 2), ket_t, ket_t)
            else:
                var = variance_of_decomp(decomp_tapered_swap, bra_ket_composite_t, Nqubits // 2 + 1, general=True)
                me  = matrix_element(get_sparse_operator(Hqub_tapered, Nqubits // 2), bra_t, ket_t)

            results[(i,j)] = (var, me)

            print(i, j, np.round(var, 6), np.round(me, 6))

In [ ]:
sum([(10**6) * np.real(list(results.values())[i][0]) for i in range(len(results.values()))])